# Day 062 — Exercise 2: parametrize_api

Testing many input variations copy-paste style is painful:

```python
assert_response(client.post('/classify', json={'score': 0.9}), 200)
assert_response(client.post('/classify', json={'score': 0.5}), 200)
assert_response(client.post('/classify', json={'score': 2.0}), 422)
# ...10 more lines
```

pytest solves this with `@pytest.mark.parametrize`. In notebooks, a `parametrize_api` helper achieves the same goal: one function, a list of cases, all results collected without crashing on the first failure.

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel, Field
from starlette.testclient import TestClient

def _build_classify_app():
    app = FastAPI()
    class _Body(BaseModel):
        score: float = Field(ge=0.0, le=1.0)
    @app.post("/classify")
    def classify(body: _Body):
        label = "high" if body.score >= 0.7 else "medium" if body.score >= 0.3 else "low"
        return {"label": label}
    return app

_client = TestClient(_build_classify_app(), raise_server_exceptions=False)


In [ ]:
from typing import Any


## Task

Implement `parametrize_api(client, cases) -> dict`:

Each `case` has `method`, `path`, `json` (optional), `expected_status`, `label`.

For each case:
- `client.request(method, path, json=json_body)`
- If `status_code != expected_status`: append `f"{label}: expected {es}, got {actual}"` to failures
- Collect all results — do NOT raise on failure

Return `{"passed": N, "failed": N, "failures": [...]}`

## Your Implementation

In [ ]:
def parametrize_api(client, cases: list[dict]) -> dict:
    """Run multiple API test cases and collect results.

    Each case is a dict with:
        method          str  — 'GET', 'POST', etc.
        path            str  — e.g. '/classify'
        json            dict|None — request body (optional)
        expected_status int  — expected HTTP status code
        label           str  — human-readable name for the case

    Returns:
        {"passed": int, "failed": int, "failures": [str, ...]}

    On status mismatch: append f"{label}: expected {es}, got {actual}" to failures.
    Do NOT raise — collect all results and return.
    """
    # TODO: iterate cases, call client.request, check status, collect pass/fail
    raise NotImplementedError


In [ ]:
def parametrize_api(client, cases: list[dict]) -> dict:
    passed, failed, failures = 0, 0, []
    for case in cases:
        label = case.get("label", f"{case['method']} {case['path']}")
        try:
            r = client.request(case["method"], case["path"],
                               json=case.get("json"))
            assert r.status_code == case["expected_status"], (
                f"{label}: expected {case['expected_status']}, got {r.status_code}")
            passed += 1
        except AssertionError as e:
            failed += 1
            failures.append(str(e))
    return {"passed": passed, "failed": failed, "failures": failures}


## Automated checks

In [ ]:
score, total = 0, 4
try:
    cases = [
        {"method": "POST", "path": "/classify", "json": {"score": 0.9},
         "expected_status": 200, "label": "high score"},
        {"method": "POST", "path": "/classify", "json": {"score": 0.5},
         "expected_status": 200, "label": "medium score"},
        {"method": "POST", "path": "/classify", "json": {"score": 0.1},
         "expected_status": 200, "label": "low score"},
        {"method": "POST", "path": "/classify", "json": {"score": 2.0},
         "expected_status": 422, "label": "out of range"},
    ]

    result = parametrize_api(_client, cases)
    assert isinstance(result, dict)
    assert "passed" in result and "failed" in result and "failures" in result
    score += 1; print("\u2705 returns dict with passed/failed/failures")

    # all 4 cases pass
    assert result["passed"] == 4, f"Expected 4 passed, got {result['passed']}"
    assert result["failed"] == 0
    score += 1; print("\u2705 all 4 valid cases pass")

    # wrong expected_status → failure is collected (not raised)
    wrong = [{"method": "POST", "path": "/classify", "json": {"score": 0.5},
              "expected_status": 404, "label": "deliberate wrong status"}]
    r2 = parametrize_api(_client, wrong)
    assert r2["failed"] == 1 and r2["passed"] == 0
    assert len(r2["failures"]) == 1
    score += 1; print("\u2705 wrong expected_status is collected as a failure (not raised)")

    # label appears in failure message
    assert "deliberate wrong status" in r2["failures"][0], (
        f"Label missing from failure: {r2['failures'][0]!r}")
    score += 1; print("\u2705 label appears in failure message")

except Exception as e:
    print(f"\u274c {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def parametrize_api(client, cases: list[dict]) -> dict:
    passed, failed, failures = 0, 0, []
    for case in cases:
        label = case.get("label", f"{case['method']} {case['path']}")
        try:
            r = client.request(case["method"], case["path"],
                               json=case.get("json"))
            assert r.status_code == case["expected_status"], (
                f"{label}: expected {case['expected_status']}, got {r.status_code}")
            passed += 1
        except AssertionError as e:
            failed += 1
            failures.append(str(e))
    return {"passed": passed, "failed": failed, "failures": failures}
```

**Why collect failures instead of raising?** Raising on the first failure means you only know about ONE broken case at a time. Collecting lets you see ALL failures in one run — e.g. 'cases 3, 7, and 11 fail' tells you much more about the regression than 'case 3 fails' followed by three re-runs.

</details>